In [1]:
import os
import pandas as pd
from pathlib import Path

## Merge all Socioeconomic

file-agnostic

In [2]:
df_socioeconomic = pd.DataFrame(columns=['Kod', 'Nazwa', 'Rok'])

path = Path('../../data/input/socioeconomic/')

for filename in os.listdir(path):
    if not filename.endswith('.csv') or 'CREL' not in filename:
        continue

    df_temp = pd.read_csv(path / filename, sep=';')

    metric_col = df_temp.columns[2]
    metrics_cols =  df_temp.columns[2:df_temp.columns.get_loc('Rok')]
    if len(metrics_cols) > 1:
        df_temp[metric_col] = df_temp.apply(lambda row: ' '.join(row[metrics_cols].astype(str).values), axis=1)

    metrics = df_temp[metric_col].unique()
    for metric in metrics:
        metric_unit = df_temp[df_temp[metric_col] == metric]['Jednostka miary'].iloc[0]
        df_values = df_temp[df_temp[metric_col] == metric][['Kod', 'Rok', 'Nazwa', 'Wartosc']]
        df_values = df_values.rename(columns={'Wartosc': f'{metric_col}: {metric} ({metric_unit})'})
        
        df_socioeconomic = df_socioeconomic.merge(df_values, on=['Kod', 'Rok', 'Nazwa'], how='outer')


df_socioeconomic = df_socioeconomic[
    (df_socioeconomic['Nazwa'].str.startswith('Powiat')) | 
    (df_socioeconomic['Nazwa'].str.isupper())
]

df_socioeconomic['Nazwa'] = df_socioeconomic['Nazwa'].apply(lambda x: x.lower() if x.isupper() else x)
df_socioeconomic['Nazwa'] = df_socioeconomic['Nazwa'].str.replace('Powiat', 'powiat').str.replace('polska', 'Polska')

df_socioeconomic['Rodzaj'] = df_socioeconomic.apply(lambda x: 'county' if 'powiat' in x['Nazwa'] \
                                                        else "country" if "Polska" in x['Nazwa'] \
                                                        else 'voivodeship', axis=1)

df_socioeconomic['Nazwa'] = df_socioeconomic['Nazwa'].replace({
    'powiat m. Wałbrzych od 2013': 'powiat m. Wałbrzych',
    'powiat m. Wałbrzych do 2002': 'powiat m. Wałbrzych',
    'powiat karkonoski': 'powiat jeleniogórski',
})
df_socioeconomic = df_socioeconomic.drop_duplicates()

df_socioeconomic['Kod'] = df_socioeconomic['Kod'].astype(str).str.zfill(7)

# fix non-numeric numeric columns
for col in df_socioeconomic.columns:
    if col not in ('Kod', 'Nazwa', 'Rodzaj') and df_socioeconomic[col].dtype == 'object':
        df_socioeconomic[col] = df_socioeconomic[col].str.replace(',', '.')
        df_socioeconomic[col] = pd.to_numeric(df_socioeconomic[col])

df_socioeconomic.reset_index(drop=True, inplace=True)
df_socioeconomic.tail()

/var/folders/yp/gjwvp9611c96kbtrz5ssff_h0000gn/T/ipykernel_28348/4253851260.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(path / filename, sep=';')


,Kod,Nazwa,Rok,Wskaźniki: ludność na 1 km2 (osoba),Rodzaje dróg: o nawierzchni twardej (km),Rodzaje dróg: o nawierzchni twardej ulepszonej (km),Rodzaje pojazdów: pojazdy samochodowe i ciągniki (szt.),Rodzaje pojazdów: samochody osobowe (szt.),Rodzaje pojazdów: autobusy ogółem (szt.),Rodzaje pojazdów: samochody ciężarowe (szt.),Rodzaje pojazdów: ciągniki siodłowe (szt.),Wypadki drogowe: wypadki ogółem (szt.),Wypadki drogowe: ofiary wypadków ogółem (osoba),Wypadki drogowe: ofiary śmiertelne (osoba),Wypadki drogowe: ranni (osoba),Typy dróg: drogi gminne i powiatowe o twardej nawierzchni na 100 km2 (km),Typy dróg: drogi gminne i powiatowe o twardej nawierzchni na 10 tys. ludności (km),Rodzaj
11558,3263000,powiat m. Świnoujście,2019,202.3,61.5,55.0,27332.0,22611.0,137.0,2569.0,117.0,34.0,NaN,3.0,36.0,49.0,24.2,county
11559,3263000,powiat m. Świnoujście,2020,199.6,63.8,58.5,28192.0,23311.0,138.0,2647.0,116.0,27.0,NaN,2.0,26.0,50.1,25.1,county
11560,3263000,powiat m. Świnoujście,2021,197.1,63.0,58.9,28783.0,23755.0,131.0,2689.0,120.0,24.0,NaN,3.0,22.0,50.4,25.6,county
11561,3263000,powiat m. Świnoujście,2022,194.8,66.6,62.6,29587.0,24336.0,136.0,2796.0,117.0,19.0,NaN,0.0,22.0,52.2,26.8,county
11562,3263000,powiat m. Świnoujście,2023,192.5,65.9,62.0,30165.0,24763.0,138.0,2864.0,126.0,29.0,32.0,1.0,31.0,56.8,29.5,county


In [3]:
column_mapping = {
    'Kod': 'id',
    'Rodzaj': 'type',
    'Nazwa': 'name',
    'Rok': 'year',
    'Wskaźniki: ludność na 1 km2 (osoba)': 'Indicators: population per 1 km2 (person)',
    'Rodzaje dróg: o nawierzchni twardej (km)': 'Road types: paved surface (km)',
    'Rodzaje dróg: o nawierzchni twardej ulepszonej (km)': 'Road types: improved paved surface (km)',
    'Rodzaje pojazdów: pojazdy samochodowe i ciągniki (szt.)': 'Vehicle types: motor vehicles and tractors (units)',
    'Rodzaje pojazdów: samochody osobowe (szt.)': 'Vehicle types: passenger cars (units)',
    'Rodzaje pojazdów: autobusy ogółem (szt.)': 'Vehicle types: total buses (units)',
    'Rodzaje pojazdów: samochody ciężarowe (szt.)': 'Vehicle types: trucks (units)',
    'Rodzaje pojazdów: ciągniki siodłowe (szt.)': 'Vehicle types: semi-trailer tractors (units)',
    'Wypadki drogowe: wypadki ogółem (szt.)': 'Road accidents: total accidents (units)',
    'Wypadki drogowe: ofiary wypadków ogółem (osoba)': 'Road accidents: total casualties (person)',
    'Wypadki drogowe: ofiary śmiertelne (osoba)': 'Road accidents: fatalities (person)',
    'Wypadki drogowe: ranni (osoba)': 'Road accidents: injured (person)',
    'Typy dróg: drogi gminne i powiatowe o twardej nawierzchni na 100 km2 (km)': 'Road types: municipal and county paved roads per 100km2 (km)',
    'Typy dróg: drogi gminne i powiatowe o twardej nawierzchni na 10 tys. ludności (km)': 'Road types: municipal and county paved roads per 10.000 inhabitants (km)'
}

df_socioeconomic = df_socioeconomic[column_mapping.keys()]
df_socioeconomic = df_socioeconomic.rename(columns=column_mapping) 
df_socioeconomic

,id,type,name,year,Indicators: population per 1 km2 (person),Road types: paved surface (km),Road types: improved paved surface (km),Vehicle types: motor vehicles and tractors (units),Vehicle types: passenger cars (units),Vehicle types: total buses (units),Vehicle types: trucks (units),Vehicle types: semi-trailer tractors (units),Road accidents: total accidents (units),Road accidents: total casualties (person),Road accidents: fatalities (person),Road accidents: injured (person),Road types: municipal and county paved roads per 100km2 (km),Road types: municipal and county paved roads per 10.000 inhabitants (km)
0,0000000,country,Polska,1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000000,country,Polska,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0000000,country,Polska,1997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0000000,country,Polska,1998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0000000,country,Polska,1999,NaN,NaN,NaN,NaN,9282816.0,78717.0,1318174.0,85013.0,NaN,NaN,6730.0,68499.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11558,3263000,county,powiat m. Świnoujście,2019,202.3,61.5,55.0,27332.0,22611.0,137.0,2569.0,117.0,34.0,NaN,3.0,36.0,49.0,24.2
11559,3263000,county,powiat m. Świnoujście,2020,199.6,63.8,58.5,28192.0,23311.0,138.0,2647.0,116.0,27.0,NaN,2.0,26.0,50.1,25.1
11560,3263000,county,powiat m. Świnoujście,2021,197.1,63.0,58.9,28783.0,23755.0,131.0,2689.0,120.0,24.0,NaN,3.0,22.0,50.4,25.6
11561,3263000,county,powiat m. Świnoujście,2022,194.8,66.6,62.6,29587.0,24336.0,136.0,2796.0,117.0,19.0,NaN,0.0,22.0,52.2,26.8


In [4]:
df_socioeconomic.to_csv('../../data/intermediate/socioeconomic.csv', index=False)